In [0]:
# ============================================================

# GOLD – dim_priority

# Grain: (source_user_id, project, priority_code)

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ---------------- CONFIG ----------------

SILVER_SLA_TABLE = "workspace.slainte_silver.sla_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_PRIORITY = f"{GOLD_DB}.dim_priority"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_sla = spark.table(SILVER_SLA_TABLE)

# ============================================================

# 1️⃣ Normalize SLA input (safe handling)

# ============================================================

df_priority_base = (

    df_sla

    .select(

        F.col("source_user_id"),

        F.col("project"),

        F.col("priority_level").cast("int").alias("priority_code")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("project").isNotNull() &

        F.col("priority_code").isNotNull()

    )

    .distinct()

)

# ============================================================

# 2️⃣ Generate surrogate key (CORRECT window)

# ============================================================

priority_window = (

    Window

    .partitionBy("source_user_id", "project")

    .orderBy("priority_code")

)

df_dim_priority = (

    df_priority_base

    .withColumn("priority_id", F.row_number().over(priority_window))

    .withColumn("created_at", F.current_timestamp())

    .select(

        "priority_id",

        "source_user_id",

        "project",

        "priority_code"

    )

)

# ============================================================

# 3️⃣ Write GOLD dimension

# ============================================================

df_dim_priority.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_PRIORITY)

# ============================================================

# 4️⃣ Preview

# ============================================================

print("✅ dim_priority created successfully")

display(

    spark.table(DIM_PRIORITY)

         .orderBy("source_user_id", "project", "priority_code")

)
 